# Classical Models and Training
Predict first, run the calculations and explain the differences. Data are generated locally with fixed seeds; reported scores describe these teaching examples, not general performance guarantees.


In [ ]:
%pip -q install "numpy>=2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4"


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error


## Independent gradient descent and a least-squares reference
The model is y=wx+b. Update both parameters using the old gradient.


In [ ]:
x=np.array([0.,1.,2.]); y=np.array([1.,3.,5.])
w=b=0.
losses=[]
for step in range(1000):
    r=w*x+b-y
    losses.append(np.mean(r*r))
    dw=2*np.mean(r*x); db=2*np.mean(r)
    w,b=w-.1*dw,b-.1*db
reference=np.linalg.lstsq(np.column_stack([x,np.ones(3)]),y,rcond=None)[0]
assert np.allclose([w,b],reference,atol=1e-7)
assert np.isclose(losses[0],35/3)
print("Fitted parameters",w,b,"reference",reference)


## A nonlinear representation for XOR
This hand calculation illustrates expressive capacity; it is not a trained network.


In [ ]:
corners=np.array([[0,0],[0,1],[1,0],[1,1]])
s=corners.sum(axis=1)
h1=np.maximum(0,s);h2=np.maximum(0,s-1)
assert np.allclose(h1-2*h2,[0,1,1,0])
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_moons
Xc,yc=make_moons(n_samples=220,noise=.18,random_state=2026)
Xct,Xce,yct,yce=train_test_split(Xc,yc,stratify=yc,test_size=.25,random_state=42)
network=make_pipeline(StandardScaler(),MLPClassifier(hidden_layer_sizes=(8,),solver="lbfgs",max_iter=3000,random_state=2026))
network.fit(Xct,yct)
print("Predeclared neural model test accuracy:",network.score(Xce,yce))


## Model selection stays inside cross-validation
Feature expansion and scaling are inside the pipeline. The test set is evaluated only after the grid search. Penalty conventions follow scikit-learn Ridge; its alpha is not numerically identical to a lambda multiplying a mean loss without a scale conversion.


In [ ]:
rng=np.random.default_rng(2026)
X=rng.uniform(-2,2,(160,1)); y=np.sin(2*X[:,0])+rng.normal(0,.2,160)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42)
pipe=make_pipeline(PolynomialFeatures(include_bias=False),StandardScaler(),Ridge())
cv=KFold(n_splits=4,shuffle=True,random_state=42)
search=GridSearchCV(pipe,{"polynomialfeatures__degree":[1,3,8],"ridge__alpha":[.01,1,100]},scoring="neg_mean_squared_error",cv=cv)
search.fit(Xtr,ytr)
print("Selected with CV:",search.best_params_)
print("Final held-out MSE:",mean_squared_error(yte,search.predict(Xte)))
assert np.allclose(search.best_estimator_[1].mean_,search.best_estimator_[0].transform(Xtr).mean(axis=0))


## Exercises
1. Increase the learning rate in the gradient calculation and inspect the loss rather than assuming convergence.
2. Compare train and validation errors across polynomial degrees.
3. Explain why the test score must not guide another round of degree selection.
